**ADITYA SINGH [MSD25020]**ADITYA SINGH [MSD25020]

In [8]:
!pip install pyspark -q

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import os
import shutil

In [3]:
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

In [5]:
import glob
import shutil

# Look for any CSV file generated inside the silver directory
silver_files = glob.glob("submissions\submissions_MSD25020_ADITYA SINGH\sales_silver.csv")
if silver_files:
    shutil.copy(silver_files[0], "/content/sales_silver.csv")
    print(f"Successfully copied clean silver file to /content/sales_silver.csv")
else:
    print("No processed silver CSV files found! Make sure you run the Spark write cell first.")

No processed silver CSV files found! Make sure you run the Spark write cell first.


<>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\ADITYA SINGH\AppData\Local\Temp\ipykernel_20532\231243538.py:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  silver_files = glob.glob("submissions\submissions_MSD25020_ADITYA SINGH\sales_silver.csv")


In [6]:
print(os.listdir("data_lake/bronze"))
print(os.listdir("data_lake/silver"))

[]
['preprocessed_sales']


### Start Spark and read the Bronze file

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
spark = (SparkSession.builder
         .appName("SalesDataLake")
         .config("spark.sql.ansi.enabled", "false")  # bad dates become null instead of errors
         .getOrCreate())

# Read from the uploaded sales_silver.csv file
raw_df = spark.read.csv("/content/sales_silver.csv", header=True, inferSchema=False)

In [ ]:
raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
raw_df.show(5, truncate=False)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710.0    |20.0            |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680.0     |5.0             |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080.0    |12.0            |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

In [ ]:
rows_before = raw_df.count()
print("Total records (before preprocessing):", rows_before)

Total records (before preprocessing): 990


In [ ]:
# Exact duplicate rows
dup_rows = rows_before - raw_df.dropDuplicates().count()
print("Exact duplicate rows:", dup_rows)

Exact duplicate rows: 0


In [ ]:
# Null / empty values in every column
raw_df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         4|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [ ]:
# Text format problems (extra spaces, mixed capital letters)
raw_df.groupBy("category").count().orderBy("category").show(30)

raw_df.groupBy("city").count().orderBy("city").show(30)

extra_space = raw_df.filter(
    (F.col("city") != F.trim(F.col("city"))) |
    (F.col("customer_name") != F.trim(F.col("customer_name")))
).count()
print("Rows with extra spaces in city/customer_name:", extra_space)

+---------------+-----+
|       category|count|
+---------------+-----+
|         Beauty|  137|
|          Books|  127|
|       Clothing|  108|
|    Electronics|  104|
|      Furniture|  129|
|        Grocery|  134|
|Home Appliances|  113|
|         Sports|  138|
+---------------+-----+

+----------+-----+
|      city|count|
+----------+-----+
| Bengaluru|   90|
|   Chennai|   88|
|Coimbatore|   81|
|     Delhi|   83|
| Hyderabad|   97|
|    Jaipur|   83|
|   Kolkata|   81|
|   Lucknow|   89|
|    Mumbai|  105|
|      Pune|   81|
|   Unknown|    8|
|Vijayawada|  104|
+----------+-----+

Rows with extra spaces in city/customer_name: 0


In [ ]:
# Invalid numbers
print("quantity <= 0       :", raw_df.filter(F.col("quantity").cast("int") <= 0).count())
print("unit_price <= 0     :", raw_df.filter(F.col("unit_price").cast("double") <= 0).count())
print("discount < 0 or >100:", raw_df.filter((F.col("discount_percent").cast("double") < 0) |
                                             (F.col("discount_percent").cast("double") > 100)).count())

quantity <= 0       : 0
unit_price <= 0     : 0
discount < 0 or >100: 0


In [ ]:
# Invalid / inconsistent dates
raw_df.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")) \
      .select("order_id", "order_date").show()

raw_df.filter(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$") &
              F.to_date("order_date", "yyyy-MM-dd").isNull()) \
      .select("order_id", "order_date").show()

+--------+----------+
|order_id|order_date|
+--------+----------+
+--------+----------+

+--------+----------+
|order_id|order_date|
+--------+----------+
+--------+----------+



### Preprocessing -> Silver layer

In [ ]:
df = raw_df

df = df.dropDuplicates()

for c in df.columns:
  df = df.withColumn(c, F.trim(F.col(c)))

for c in ["customer_name", "category", "city", "state"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

# Missing values
for c in ["customer_name", "payment_method", "city"]:
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
df = df.fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

# Convert numeric columns to numbers
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

In [ ]:
df = df.withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
df = df.withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
df = df.withColumn("discount_percent",
                   F.when((F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
                          F.col("discount_percent")).otherwise(0.0))

overall_qty_median = df.approxQuantile("quantity", [0.5], 0.01)[0]
overall_price_median = df.approxQuantile("unit_price", [0.5], 0.01)[0]

product_window = Window.partitionBy("product")
product_price_median = F.percentile_approx("unit_price", 0.5).over(product_window)

df = df.withColumn("unit_price",
                   F.coalesce(F.col("unit_price"), product_price_median, F.lit(overall_price_median)))
df = df.withColumn("quantity",
                   F.coalesce(F.col("quantity"), F.lit(int(overall_qty_median))))

In [ ]:
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "yyyy/MM/dd"),
        F.to_date("order_date", "dd/MM/yyyy"),
        F.to_date("order_date", "dd-MM-yyyy"),
        F.to_date("order_date", "MM-dd-yyyy"),
    )
)

print("Rows with invalid date set to null:", df.filter(F.col("order_date").isNull()).count())

Rows with invalid date set to null: 4


In [ ]:
df = df.orderBy("order_id")

silver_path = "data_lake/silver/preprocessed_sales"
df.coalesce(1).write.mode("overwrite").option("header", True).csv(silver_path)

### Verification

In [ ]:
rows_after = df.count()
print("Rows BEFORE preprocessing:", rows_before)
print("Rows AFTER  preprocessing:", rows_after)
print("Rows removed             :", rows_before - rows_after)

Rows BEFORE preprocessing: 990
Rows AFTER  preprocessing: 990
Rows removed             : 0


In [ ]:
silver_df = spark.read.csv(silver_path, header=True, inferSchema=True)
print("\nSilver data (read back)")
silver_df.printSchema()
silver_df.show(10, truncate=False)
print("Silver record count:", silver_df.count())


Silver data (read back)
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+-------

In [ ]:
print("Categories now:")
silver_df.groupBy("category").count().show()

Categories now:
+---------------+-----+
|       category|count|
+---------------+-----+
|         Sports|  138|
|        Grocery|  134|
|    Electronics|  104|
|       Clothing|  108|
|          Books|  127|
|Home Appliances|  113|
|      Furniture|  129|
|         Beauty|  137|
+---------------+-----+



In [ ]:
print("Bronze folder:", os.listdir("data_lake/bronze"))

Bronze folder: []
